In [1]:
from google.colab import drive
drive.mount('/content/drive')

import gc
# gc.collect()

Mounted at /content/drive


Reading pdfs

In [ ]:
!pip install PyMuPDF

In [ ]:
import os

pdf_folder = '/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/1. Corelaws'
output_folder = '/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/'
merged_file_path = os.path.join(output_folder, 'merged_pdf_text.txt')

# List all PDFs
pdf_files = [os.path.join(pdf_folder, f) for f in os.listdir(pdf_folder) if f.endswith('.pdf')]

print(f"Found {len(pdf_files)} PDFs")

Found 63 PDFs


In [ ]:
import re

def clean_text(text):
    text = re.sub(r'\n+', '\n', text)        # Replace multiple newlines
    text = re.sub(r' +', ' ', text)          # Replace multiple spaces
    text = re.sub(r'Page \d+', '', text)     # Remove page numbers
    return text.strip()

In [ ]:
import fitz  # PyMuPDF

with open(merged_file_path, "w", encoding="utf-8") as merged_file:
    for pdf_file in pdf_files:
        doc = fitz.open(pdf_file)
        merged_file.write(f"\n=== {os.path.basename(pdf_file)} ===\n\n")  # separator

        for page in doc:
            text = page.get_text()
            if text:
                merged_file.write(clean_text(text) + "\n\n")

        doc.close()
        print(f"Processed: {os.path.basename(pdf_file)}")

print(f"\n✅ All PDFs merged into: {merged_file_path}")

In [ ]:
merged_file_path = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/merged_pdf_text.txt"

with open(merged_file_path, "r", encoding="utf-8") as f:
    content = f.read()  # be careful if file is huge
    print("Total characters:", len(content))

Total characters: 329277909


Reading XML FILES

In [ ]:
xml_folder = '/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/2. CFR-2025'
xml_files = []

for root, dirs, files in os.walk(xml_folder):
    for file in files:
        if file.endswith('.xml'):
            xml_files.append(os.path.join(root, file))

print(f"Found {len(xml_files)} XML files")

Found 180 XML files


In [ ]:
import xml.etree.ElementTree as ET

xml_file = '/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/2. CFR-2025/title-1/CFR-2025-title1-vol1.xml'  # example

tree = ET.parse(xml_file)
root = tree.getroot()

def print_xml_elements(element, level=0):
    indent = "  " * level
    print(f"{indent}{element.tag}: {element.text.strip() if element.text else ''}")
    for child in element:
        print_xml_elements(child, level+1)

print_xml_elements(root)

In [ ]:
import os
import csv
import xml.etree.ElementTree as ET

xml_folder = r"/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/2. CFR-2025"
output_csv_xml = r"/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/2.1_xml_texts.csv"

xml_files = [os.path.join(root, f) for root, dirs, files in os.walk(xml_folder) for f in files if f.endswith('.xml')]
print(f"Found {len(xml_files)} XML files")

def extract_xml_text(root):
    """
    Extract headers (HD) and paragraph text (P) from a CFR XML.
    Returns a list of tuples: (header, paragraph_text)
    """
    data = []
    current_header = ""
    for elem in root.iter():
        if elem.tag == "HD" and elem.text and elem.text.strip():
            current_header = elem.text.strip()
        elif elem.tag == "P" and elem.text and elem.text.strip():
            paragraph = elem.text.strip()
            data.append((current_header, paragraph))
    return data

# Process all XML files
xml_data = []
for xml_file in xml_files:
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        sections = extract_xml_text(root)
        for title, text in sections:
            xml_data.append([os.path.basename(xml_file), title, text])
    except Exception as e:
        print(f"Error reading {xml_file}: {e}")

# Save to CSV
with open(output_csv_xml, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['filename', 'section_title', 'section_text'])
    writer.writerows(xml_data)

print(f"XML sections saved to {output_csv_xml}")

Found 180 XML files
XML sections saved to /content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/2.1_xml_texts.csv


In [ ]:
import pandas as pd

df = pd.read_csv(output_csv_xml)
df.head(6)

,filename,section_title,section_text
0,CFR-2025-title1-vol1.xml,NaN,Published by the Office of the Federal Registe...
1,CFR-2025-title1-vol1.xml,Legal Status and Use of Seals and Logos,The seal of the National Archives and Records ...
2,CFR-2025-title1-vol1.xml,Legal Status and Use of Seals and Logos,It is prohibited to use NARA's official seal a...
3,CFR-2025-title1-vol1.xml,Use of ISBN Prefix,This is the Official U.S. Government edition o...
4,CFR-2025-title1-vol1.xml,Use of ISBN Prefix,U . S . G O V E R N M E N T P U B L I S H I N ...
5,CFR-2025-title1-vol1.xml,Use of ISBN Prefix,"U.S. Superintendent of Documents • Washington,..."


# DATASET 3

In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/3. opinions-2023-04-30.csv.bz2"

# Read only the first row to see column names
sample = pd.read_csv(file_path, compression='bz2', nrows=5)
print(sample.columns)   # prints all column names
print(sample.tail(1))    # optional: see first 5 rows

In [ ]:
# import pandas as pd
# import os
# from bs4 import BeautifulSoup
# import itertools
# import re


# # --- CONFIG ---
# file_path = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/3. opinions-2023-04-30.csv.bz2"
# output_dir = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/3.1 processed_chunks"
# chunksize = 20000  # rows per chunk to avoid memory crash

# # Columns to keep (metadata)
# columns_to_keep = [
#     'type', 'sha1', 'page_count', 'plain_text', 'html', 'html_lawbox',
#     'html_columbia', 'html_anon_2020', 'xml_harvard', 'html_with_citations',
#     'extracted_by_ocr', 'cluster_id'
# ]

# os.makedirs(output_dir, exist_ok=True)

# # --- PROCESS IN CHUNKS ---
# chunk_iter = pd.read_csv(file_path, compression='bz2', usecols=columns_to_keep, chunksize=chunksize, low_memory=False)

# existing_files = os.listdir(output_dir)

# # Extract already processed chunk numbers
# processed_chunks = set()
# for f in existing_files:
#     if f.startswith("chunk_") and f.endswith(".parquet"):
#         idx = int(f.split("_")[1].split(".")[0])
#         processed_chunks.add(idx)

# print(f"Already processed chunks: {sorted(processed_chunks)}")

# for i, chunk in enumerate(chunk_iter):

#     # Skip already processed chunks
#     if i in processed_chunks:
#         print(f"Skipping chunk {i}, already processed")
#         continue

#     # --- CREATE A UNIFIED TEXT COLUMN ---
#     chunk['text'] = chunk['plain_text']  # start with plain_text
#     chunk['text'] = chunk['text'].fillna(chunk['xml_harvard'])
#     chunk['text'] = chunk['text'].fillna(chunk['html_lawbox'])
#     chunk['text'] = chunk['text'].fillna(chunk['html'])

#     # --- CLEAN TEXT (remove HTML/XML tags) ---
#     chunk['text'] = chunk['text'].apply(
#         lambda x: BeautifulSoup(str(x), "html.parser").get_text() if pd.notnull(x) else None
#     )

#     # --- DROP ORIGINAL HEAVY TEXT COLUMNS (IMPORTANT) ---
#     drop_cols = [
#         'plain_text', 'html', 'html_lawbox',
#         'html_columbia', 'html_anon_2020',
#         'xml_harvard', 'html_with_citations',
#         'type',	'sha1',	'page_count',
#         'extracted_by_ocr',	'cluster_id'
#     ]

#     chunk.drop(columns=[col for col in drop_cols if col in chunk.columns], inplace=True)

#     # --- DROP COLUMNS WITH ALL NULL VALUES ---
#     chunk.dropna(axis=1, how='all', inplace=True)

#     # --- OPTIONAL: DROP ROWS WHERE TEXT IS EMPTY ---
#     chunk = chunk[chunk['text'].notnull() & (chunk['text'].str.strip() != "")]

#     # Save cleaned chunk
#     chunk.to_parquet(os.path.join(output_dir, f"chunk_{i}.parquet"), index=False)

#     del chunk
#     gc.collect()

#     print(f"Processed chunk {i}, rows: {len(chunk)}, saved at chunk_{i}.parquet")

# print("All chunks processed! Text content is now ready for analysis.")

In [2]:
import pandas as pd
import os
import gc
import re

# --- CONFIG ---
file_path = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/3. opinions-2023-04-30.csv.bz2"
output_dir = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/3.1 processed_chunks"

chunksize = 20000

columns_to_keep = [
    'type', 'sha1', 'page_count', 'plain_text', 'html', 'html_lawbox',
    'html_columbia', 'html_anon_2020', 'xml_harvard',
    'html_with_citations', 'extracted_by_ocr', 'cluster_id'
]

os.makedirs(output_dir, exist_ok=True)

# --- FAST HTML/XML STRIP ---
def clean_text(text):
    if text is None or pd.isna(text):
        return None
    text = str(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# --- FIND EXISTING CHUNKS ---
existing_files = os.listdir(output_dir)

processed_chunks = set()
for f in existing_files:
    if f.startswith("chunk_") and f.endswith(".parquet"):
        idx = int(f.split("_")[1].split(".")[0])
        processed_chunks.add(idx)

print(f"Already processed chunks: {sorted(processed_chunks)}")

# --- READ CSV IN CHUNKS ---
chunk_iter = pd.read_csv(
    file_path,
    compression='bz2',
    usecols=columns_to_keep,
    chunksize=chunksize,
    low_memory=False
)

for i, chunk in enumerate(chunk_iter):

    if i in processed_chunks:
        print(f"Skipping chunk {i}")
        continue

    print(f"Processing chunk {i}...")

    # --- BUILD TEXT (NO EXTRA LIST VARIABLE) ---
    def pick_text(a, b, c, d):
      for val in (a, b, c, d):
          if val is not None and not pd.isna(val):
              return clean_text(val)
      return None

    chunk['text'] = [
        pick_text(a, b, c, d)
        for a, b, c, d in zip(
            chunk['plain_text'],
            chunk['xml_harvard'],
            chunk['html_lawbox'],
            chunk['html']
        )
    ]

    # --- KEEP ONLY NEEDED COLUMNS ---
    chunk = chunk[['text']]

    # --- DROP EMPTY TEXT ROWS ---
    mask = chunk['text'].notnull() & (chunk['text'].str.len() > 0)
    chunk = chunk.loc[mask].copy()

    # --- SAVE ---
    out_path = os.path.join(output_dir, f"chunk_{i}.parquet")
    chunk.to_parquet(out_path, index=False)

    print(f"Saved chunk {i}, rows: {len(chunk)}")

    # --- MEMORY CLEANUP ---
    del chunk
    gc.collect()

print("All chunks processed successfully.")

Already processed chunks: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216